In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-23 17:01:58,292 | INFO | EasyLLM 初始化完成: provider=anthropic_native, model=qwen3.5-9b
2026-04-23 17:01:58,444 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: anthropic_native


In [ ]:
test_invoke_without_tool(agent)

In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [ ]:
await test_astream_without_tool(agent)

In [3]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-23 17:02:02,873 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-23 17:02:02,874 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-23 17:02:02,874 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-23 17:02:02,875 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [7]:
test_invoke_with_tool(agent)

2026-04-21 18:06:12,390 | INFO | 对话历史已清空
2026-04-21 18:06:12,390 | INFO | 使用工具模式调用智能体
2026-04-21 18:06:14,535 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"
2026-04-21 18:06:14,537 | INFO | 思考内容: 用户要求我：
1. 使用工具翻译"你是谁，在哪里"到英语
2. 用计算器计算 3^22

我需要调用两个工具：
1. translate_tool 翻译中文到英语
2. calculator 计算 3^22

让我先并行调用这两个工具。

2026-04-21 18:06:14,538 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你是谁，在哪里', 'target_lang': 'English'}
2026-04-21 18:06:14,538 | INFO | test_skill执行工具: calculator，参数: {'expression': '3**22'}
2026-04-21 18:06:17,415 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"
2026-04-21 18:06:17,417 | INFO | 思考内容: 用户要求我：
1. 翻译"你是谁，在哪里"到英语
2. 计算 3^22

我已经调用了两个工具。现在需要分析结果：

1. 翻译结果看起来很奇怪，它返回了"Translated: 你是谁，在哪里"，这看起来没有正确翻译。这可能是因为工具返回的格式问题，或者翻译功能有问题。

2. 计算器结果：3^22 = 31381059609，这个看起来是正确的。

我需要告诉用户：
- 翻译结果（虽然看起来有问题）
- 计算结果
- 指出翻译工具可能存在问题

2026-04-21 18:06:17,418 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你



翻译工具**未能正常工作**，两次调用都返回了原始中文，没有翻译结果。

计算结果**正确**：3²² = **31,381,059,609**

---

**总结：**
| 任务 | 状态 |
|------|------|
| 翻译到英语 | ❌ 工具无响应 |
| 计算 3²² | ✅ 31,381,059,609 |

翻译工具似乎存在故障，建议重新检查或联系服务提供方。


In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [4]:
await test_astream_with_tool(agent)

2026-04-23 17:02:05,701 | INFO | 对话历史已清空
2026-04-23 17:02:05,760 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


round 1

thinking content:
用户要求我：
1. 使用工具翻译"你是谁，在哪里"到英语
2. 判断翻译工具是否正确
3. 计算 3^22

我需要先调用工具来完成这两个任务，然后检查结果并回应用户。

让我先调用翻译工具和计算器工具。

content:



tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': '英语'}
calculator : {'expression': '3**22'}


2026-04-23 17:02:07,871 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"



round 2

thinking content:
用户要求我：
1. 将"你是谁，在哪里"翻译成英语，并判断翻译工具是否正确
2. 计算 3^22

从工具返回结果来看：
1. 翻译工具返回的结果是"Translated: 你是谁，在哪里"，这显然是错误的！它没有真正翻译，只是返回了原文。正确的翻译应该是 "Who are you, where are you" 或类似表达。
2. 计算器返回了 31381059609，这是 3^22 的正确结果。

我需要告诉用户翻译工具出错了，并提供正确的翻译，同时给出计算结果。

content:


翻译工具**没有正确工作**。返回的结果是原文，而不是英文翻译。

**正确的翻译应该是：**
- "Who are you, where are you"

**计算结果：**
- 3²² = 31,381,059,609

---

**问题总结：**
| 任务 | 状态 | 说明 |
|------|------|------|
| 翻译 | ❌ 失败 | 工具返回了原文而非英文 |
| 计算 | ✅ 正确 | 3^22 = 31,381,059,609 |

建议重新使用翻译工具，或手动翻译该句子。
final res:


翻译工具**没有正确工作**。返回的结果是原文，而不是英文翻译。

**正确的翻译应该是：**
- "Who are you, where are you"

**计算结果：**
- 3²² = 31,381,059,609

---

**问题总结：**
| 任务 | 状态 | 说明 |
|------|------|------|
| 翻译 | ❌ 失败 | 工具返回了原文而非英文 |
| 计算 | ✅ 正确 | 3^22 = 31,381,059,609 |

建议重新使用翻译工具，或手动翻译该句子。


/home/wxd/.local/lib/python3.10/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ParsedTextBlock[~ResponseFormatT]` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=No

In [5]:
raw_history=agent.get_raw_history()  
raw_history

[{'role': 'user',
  'content': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22'},
 {'role': 'assistant',
  'content': [{'type': 'thinking',
    'thinking': '用户要求我：\n1. 使用工具翻译"你是谁，在哪里"到英语\n2. 判断翻译工具是否正确\n3. 计算 3^22\n\n我需要先调用工具来完成这两个任务，然后检查结果并回应用户。\n\n让我先调用翻译工具和计算器工具。\n',
    'signature': 'a849f9657eb447efa014f523e43378c7'},
   {'type': 'text', 'text': '\n\n'},
   {'type': 'tool_use',
    'id': 'call_68e9f91b5f404cffac878134',
    'name': 'translate_tool',
    'input': {'text': '你是谁，在哪里', 'target_lang': '英语'}},
   {'type': 'tool_use',
    'id': 'call_de5d5e6e5f4847ebb1725c60',
    'name': 'calculator',
    'input': {'expression': '3**22'}}],
  'reasoning_content': '用户要求我：\n1. 使用工具翻译"你是谁，在哪里"到英语\n2. 判断翻译工具是否正确\n3. 计算 3^22\n\n我需要先调用工具来完成这两个任务，然后检查结果并回应用户。\n\n让我先调用翻译工具和计算器工具。\n'},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'call_68e9f91b5f404cffac878134',
    'content': 'Translated: 你是谁，在哪里',
    'name': 'translate_tool'},
   {'type': 'tool_result',
    'too

In [6]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

2026-04-23 17:02:19,638 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b


In [7]:
raw_history2=agent.get_raw_history()  

In [8]:
raw_history2

[{'role': 'user',
  'content': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22'},
 {'role': 'assistant',
  'content': '\n\n',
  'tool_calls': [{'id': 'call_68e9f91b5f404cffac878134',
    'type': 'function',
    'function': {'name': 'translate_tool',
     'arguments': '{"text": "你是谁，在哪里", "target_lang": "英语"}'}},
   {'id': 'call_de5d5e6e5f4847ebb1725c60',
    'type': 'function',
    'function': {'name': 'calculator',
     'arguments': '{"expression": "3**22"}'}}],
  'reasoning_content': '用户要求我：\n1. 使用工具翻译"你是谁，在哪里"到英语\n2. 判断翻译工具是否正确\n3. 计算 3^22\n\n我需要先调用工具来完成这两个任务，然后检查结果并回应用户。\n\n让我先调用翻译工具和计算器工具。\n'},
 {'role': 'tool',
  'content': 'Translated: 你是谁，在哪里',
  'tool_call_id': 'call_68e9f91b5f404cffac878134',
  'name': 'translate_tool'},
 {'role': 'tool',
  'content': '31381059609',
  'tool_call_id': 'call_de5d5e6e5f4847ebb1725c60',
  'name': 'calculator'},
 {'role': 'assistant',
  'content': '\n\n翻译工具**没有正确工作**。返回的结果是原文，而不是英文翻译。\n\n**正确的翻译应该是：**\n- "Who are you, where are you"\n\n**计算结果：**\

In [8]:
raw_history==raw_history2

False

In [9]:
await agent.astream_invoke(f"我们刚才聊了什么")


round 1


2026-04-23 17:02:33,737 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



thinking content:
用户问我们刚才聊了什么，我需要回顾一下对话历史：

1. 用户请求我：
   - 翻译"你是谁，在哪里"到英语
   - 计算 3^22

2. 我调用了两个工具：
   - 翻译工具：返回了原文而非英文翻译（工具失败）
   - 计算器：正确返回了 31381059609

3. 我指出了翻译工具的问题，并给出了正确翻译和计算结果

现在用户问我们聊了什么，我应该简洁地总结对话内容。

content:


我们刚才讨论了两个任务：

1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语
   - 翻译工具未能正常工作，返回了原文
   - 正确翻译应为："Who are you, where are you"

2. **计算任务**：计算 3 的 22 次方
   - 结果：**31,381,059,609**

需要我继续帮你做什么吗？
final res:


我们刚才讨论了两个任务：

1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语
   - 翻译工具未能正常工作，返回了原文
   - 正确翻译应为："Who are you, where are you"

2. **计算任务**：计算 3 的 22 次方
   - 结果：**31,381,059,609**

需要我继续帮你做什么吗？


'\n\n我们刚才讨论了两个任务：\n\n1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语\n   - 翻译工具未能正常工作，返回了原文\n   - 正确翻译应为："Who are you, where are you"\n\n2. **计算任务**：计算 3 的 22 次方\n   - 结果：**31,381,059,609**\n\n需要我继续帮你做什么吗？'

In [10]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

2026-04-23 17:02:44,066 | INFO | EasyLLM 初始化完成: provider=anthropic_native, model=qwen3.5-9b


In [11]:
raw_history3=agent.get_raw_history()  
raw_history3

[{'role': 'user',
  'content': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22'},
 {'role': 'assistant',
  'content': [{'type': 'thinking',
    'thinking': '用户要求我：\n1. 使用工具翻译"你是谁，在哪里"到英语\n2. 判断翻译工具是否正确\n3. 计算 3^22\n\n我需要先调用工具来完成这两个任务，然后检查结果并回应用户。\n\n让我先调用翻译工具和计算器工具。\n',
    'signature': 'a849f9657eb447efa014f523e43378c7'},
   {'type': 'text', 'text': '\n\n'},
   {'type': 'tool_use',
    'id': 'call_68e9f91b5f404cffac878134',
    'name': 'translate_tool',
    'input': {'text': '你是谁，在哪里', 'target_lang': '英语'}},
   {'type': 'tool_use',
    'id': 'call_de5d5e6e5f4847ebb1725c60',
    'name': 'calculator',
    'input': {'expression': '3**22'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'call_68e9f91b5f404cffac878134',
    'content': 'Translated: 你是谁，在哪里',
    'name': 'translate_tool'},
   {'type': 'tool_result',
    'tool_use_id': 'call_de5d5e6e5f4847ebb1725c60',
    'content': '31381059609',
    'name': 'calculator'}]},
 {'role': 'assistant',
  'content':

In [12]:
await agent.astream_invoke(f"我们刚才聊了什么")


2026-04-23 17:03:06,189 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


round 1

thinking content:
用户又问了一遍"我们刚才聊了什么"，这和我刚才回答的内容一样。我需要简洁地总结我们的对话内容。

让我回想一下：
1. 用户请求将中文"你是谁，在哪里"翻译成英语，并计算 3^22
2. 我调用了翻译工具，但翻译结果不正确（返回了原文）
3. 计算工具正确返回了 31381059609
4. 我总结了翻译失败和计算成功的情况
5. 然后用户又问"我们刚才聊了什么"

我应该简洁地总结对话内容。

content:


我们刚才的对话内容：

### 1. 翻译任务
- 用户请求将 **"你是谁，在哪里"** 翻译成英语
- 翻译工具**未能正常工作**，返回了原文而非英文
- 正确翻译应为：**"Who are you, where are you"**

### 2. 计算任务
- 用户要求计算 **3²²**
- 计算结果：**31,381,059,609**

需要我继续帮助吗？
final res:


我们刚才的对话内容：

### 1. 翻译任务
- 用户请求将 **"你是谁，在哪里"** 翻译成英语
- 翻译工具**未能正常工作**，返回了原文而非英文
- 正确翻译应为：**"Who are you, where are you"**

### 2. 计算任务
- 用户要求计算 **3²²**
- 计算结果：**31,381,059,609**

需要我继续帮助吗？


'\n\n我们刚才的对话内容：\n\n### 1. 翻译任务\n- 用户请求将 **"你是谁，在哪里"** 翻译成英语\n- 翻译工具**未能正常工作**，返回了原文而非英文\n- 正确翻译应为：**"Who are you, where are you"**\n\n### 2. 计算任务\n- 用户要求计算 **3²²**\n- 计算结果：**31,381,059,609**\n\n需要我继续帮助吗？'

In [13]:
from dotenv import load_dotenv
load_dotenv()
llm3=EasyLLM()

2026-04-23 17:03:57,069 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash


In [14]:
agent.change_model(llm=llm3)

In [15]:
history4=agent.get_raw_history()
history4

[{'role': 'user',
  'parts': [{'text': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22'}]},
 {'role': 'model',
  'parts': [{'text': '用户要求我：\n1. 使用工具翻译"你是谁，在哪里"到英语\n2. 判断翻译工具是否正确\n3. 计算 3^22\n\n我需要先调用工具来完成这两个任务，然后检查结果并回应用户。\n\n让我先调用翻译工具和计算器工具。\n',
    'thought': True},
   {'text': '\n\n'},
   {'function_call': {'id': 'call_68e9f91b5f404cffac878134',
     'name': 'translate_tool',
     'args': {'text': '你是谁，在哪里', 'target_lang': '英语'}}},
   {'function_call': {'id': 'call_de5d5e6e5f4847ebb1725c60',
     'name': 'calculator',
     'args': {'expression': '3**22'}}}]},
 {'role': 'user',
  'parts': [{'function_response': {'id': 'call_68e9f91b5f404cffac878134',
     'name': 'translate_tool',
     'response': {'result': 'Translated: 你是谁，在哪里'}}},
   {'function_response': {'id': 'call_de5d5e6e5f4847ebb1725c60',
     'name': 'calculator',
     'response': {'result': '31381059609'}}}]},
 {'role': 'model',
  'parts': [{'text': '用户要求我：\n1. 将"你是谁，在哪里"翻译成英语，并判断翻译工具是否正确\n2. 计算 3^22\n\n从工具返回结果来看：\n1. 翻译工

In [19]:
await agent.astream_invoke(f"我们刚才聊了什么")

round 1

thinking content:
**Considering User Repetition**

I'm focusing on the user's repeated question, "What did we just talk about?". My current thinking is that this might be a test or a deliberate pattern. I'm aiming for a brief, clear response summarizing the previous exchange while acknowledging the user's iterative query.


**Reiterating Prior Discussion**

I've just been asked again, "What did we just talk about?". I'm summarizing: The previous conversation involved a translation failure and a calculation. Specifically, a Chinese translation request failed initially, and a calculation of 3 to the power of 22 was computed. I'll maintain brevity.



content:
我们刚才主要聊了以下两件事：

1.  **翻译与工具验证**：你要求将“你是谁，在哪里”翻译成英语。我发现翻译工具返回了原文（未成功翻译），并指出正确的翻译应为 "Who are you, where are you"。
2.  **数学计算**：我为你计算了 **3²²**，结果是 **31,381,059,609**。

如果你有其他问题或需要重新尝试翻译，请告诉我。
final res:
我们刚才主要聊了以下两件事：

1.  **翻译与工具验证**：你要求将“你是谁，在哪里”翻译成英语。我发现翻译工具返回了原文（未成功翻译），并指出正确的翻译应为 "Who are you, where are you"。
2.  **数学计算**：

'我们刚才主要聊了以下两件事：\n\n1.  **翻译与工具验证**：你要求将“你是谁，在哪里”翻译成英语。我发现翻译工具返回了原文（未成功翻译），并指出正确的翻译应为 "Who are you, where are you"。\n2.  **数学计算**：我为你计算了 **3²²**，结果是 **31,381,059,609**。\n\n如果你有其他问题或需要重新尝试翻译，请告诉我。'

In [16]:
agent2=BasicAgent.load_session("1222",llm=llm,tool_registry=agent.tool_registry,skill_manager=agent.skill_manager)

2026-04-18 23:35:42,405 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 启用，provider: anthropic_native
2026-04-18 23:35:42,407 | INFO | 会话已恢复: 1222


In [18]:
agent2.get_history()==agent.get_history()

True